# 05 · Top-K Feature Subset — 단순화로 동등 성능 (EXP #29, LB 0.6906)

직전 = #17 T-AX C1(V3 26 feat, LB 0.6906). feature가 26개로 많고, gain 하위(rank 11~26)는 plateau + 대수적 중복이 의심된다.

**가정**: gain 상위 K개만 남긴 subset이 V3 26과 OOF 동등 이상일 수 있다. 특히 V3 26엔 **대수적 종속**(`a_tang = va_dot×acc_norm`, `acc_norm² = a_tang²+a_cent²`, `a_cent² = a_N²+a_B²` → 6 feat 자유도 3)이 있어, 중복 제거가 성능을 깎지 않을 것이다.

**실험**: gain 기준 nested subset(K5/10/15/20/26) + 대수 ablation 2개(K5_indep=K5−va_dot, K_tied_min=K26−{va_dot,acc_norm})를 3 seed로 비교(540 model). K26이 #17 재현인지 sanity 후 3-gate + 4th guard.

**결과**: **K15가 LB 동률(±0.0000)로 채택** — 26→15로 42% 단순화하며 무손실(Karpathy simplicity tiebreak가 LB로 정당). 한편 **대수적 중복 ≠ LGBM 중복**(K5_indep<K5): LGBM은 종속 feature도 다른 split에서 활용. 이후 모든 lever의 새 baseline.

## 셀 0 — imports + 상수

In [ ]:
import os, time, json, math
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01

SEEDS = [42, 7, 123]
BEST_MAX_W = 2.5
# T-AX C1 axis-wise σ (Frenet std 1:1 proportional)
SIGMA_C1 = (0.0043, 0.0035, 0.0027)  # (σ_L, σ_N, σ_B)

# T-AX C1 reference (EXP #17, plan §0)
C1_REF_OVERALL = 0.6622
C1_REF_OVERALL_STD = 0.0010
C1_REF_MAJOR = 0.7295
C1_REF_MINOR = 0.3022
C1_REF_P1 = 0.6607

os.makedirs('results', exist_ok=True)
print(f'SEEDS={SEEDS}, σ_C1=({SIGMA_C1[0]*1000:.1f},{SIGMA_C1[1]*1000:.1f},{SIGMA_C1[2]*1000:.1f})mm, max_w={BEST_MAX_W}')
print(f'T-AX C1 reference: overall {C1_REF_OVERALL}±{C1_REF_OVERALL_STD}, P1 {C1_REF_P1}')

## 셀 1 — helper (T-AX 동일)

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def w_gaussian_one(e, sigma, max_w, r=HIT_RADIUS):
    return np.clip(1.0 + (max_w - 1.0) * np.exp(-(e - r)**2 / (2 * sigma**2)),
                   0.5, max_w)


def w_axis_3(e_cart, sigmas, max_w):
    return np.stack([
        w_gaussian_one(e_cart, sigmas[0], max_w),
        w_gaussian_one(e_cart, sigmas[1], max_w),
        w_gaussian_one(e_cart, sigmas[2], max_w),
    ], axis=1).astype(np.float32)


def make_splits(minority_int, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(minority_int)), minority_int)]


lgbm_params = dict(
    objective='regression_l1', metric='mae',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=5,
    max_bin=511, n_estimators=500, random_state=42, verbose=-1,
)
print('helper 정의 완료')

## 셀 2 — Drive mount + 데이터 로드 (캐시 우선)

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

DATA_DIR = None
if not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS)):
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    !unzip -q -o "{ZIP_PATH}" -d /content/

def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv',
              f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')

DATA_DIR = _resolve_data_dir()
if DATA_DIR is not None:
    print(f'DATA_DIR = {DATA_DIR}')
else:
    print('cache 존재 — DATA_DIR resolve skip')

if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)

if DATA_DIR is None:
    DATA_DIR = _resolve_data_dir()
    if DATA_DIR is None:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        !unzip -q -o /content/drive/MyDrive/open.zip -d /content/
        DATA_DIR = _resolve_data_dir()
        assert DATA_DIR is not None, 'DATA_DIR resolve fail after unzip'

sample_sub_path = _resolve_sample_sub(DATA_DIR)
print(f'sample_submission: {sample_sub_path}')
sample_sub = pd.read_csv(sample_sub_path)
test_ids = sample_sub['id'].tolist()

if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}')

## 셀 3 — base + Frenet frame + kinematics (T-AX 동일)

In [ ]:
def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
residual_cart_train = (y_train - base_train).astype(np.float32)
vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)

r_L = (residual_cart_train * eL_tr).sum(1)
r_N = (residual_cart_train * eN_tr).sum(1)
r_B = (residual_cart_train * eB_tr).sum(1)
rec = r_L[:, None]*eL_tr + r_N[:, None]*eN_tr + r_B[:, None]*eB_tr
inv_err = np.abs(rec - residual_cart_train).max()
assert inv_err < 1e-5
residual_fren_train = np.stack([r_L, r_N, r_B], axis=1).astype(np.float32)
print(f'frame inverse max err: {inv_err:.2e}')
print(f'residual_fren std L:N:B = {residual_fren_train.std(axis=0)*1000} mm')

acc_norm_last_tr = _norm(al_tr)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
print(f'minority: {minority_mask_tr.sum()}/{len(minority_mask_tr)}')

## 셀 4 — V3 26 feat builder + FEAT_NAMES (T-AX 동일)

In [ ]:
FEAT_NAMES_V3_26 = [
    'speed_last','acc_norm_last','acc_norm_w3',
    'vx_std','vy_std','vz_std','ax_std','ay_std','az_std',
    'path_eff','distance_r','radial_v',
    'turn_mean','cos_turn_last','va_dot',
    'a_tang_last','a_cent_last','speed_diff_half','turn_mean_half_diff',
    'a_N','a_B','j_L','j_N','j_B','aw_L','aw_N'
]
assert len(FEAT_NAMES_V3_26) == 26

def build_v3_26_full(traj, eL, eN, eB):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    a_tang_last = (v_last * a_last).sum(1) / (speed_last + 1e-9)
    a_cent_last = _norm(np.cross(v_last, a_last)) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    a_N  = (a_last * eN).sum(1); a_B = (a_last * eB).sum(1)
    j_L  = (jerk_last * eL).sum(1); j_N = (jerk_last * eN).sum(1); j_B = (jerk_last * eB).sum(1)
    aw_L = (a_w3 * eL).sum(1); aw_N = (a_w3 * eN).sum(1)
    X = np.column_stack([
        speed_last, acc_norm_last, acc_norm_w3,
        v_std[:,0], v_std[:,1], v_std[:,2], a_std[:,0], a_std[:,1], a_std[:,2],
        path_eff, distance_r, radial_v, turn_mean, cos_turn_last, va_dot,
        a_tang_last, a_cent_last, speed_diff_half, turn_mean_half_diff,
        a_N, a_B, j_L, j_N, j_B, aw_L, aw_N
    ]).astype(np.float32)
    return X

X_tr_full = build_v3_26_full(traj_train, eL_tr, eN_tr, eB_tr)
X_ts_full = build_v3_26_full(traj_test,  eL_ts, eN_ts, eB_ts)
assert X_tr_full.shape == (10000, 26) and X_ts_full.shape == (10000, 26)
print(f'X_tr_full {X_tr_full.shape}, X_ts_full {X_ts_full.shape}')

## 셀 5 — Algebraic dependency sanity (V3 26 redundancy 정량 재확인)

(1) a_tang_last = va_dot × acc_norm_last  
(2) acc_norm_last² = a_tang_last² + a_cent_last²  
(3) a_cent_last² = a_N² + a_B²

In [ ]:
_idx = {n: i for i, n in enumerate(FEAT_NAMES_V3_26)}
_at = X_tr_full[:, _idx['a_tang_last']]
_va = X_tr_full[:, _idx['va_dot']]
_an = X_tr_full[:, _idx['acc_norm_last']]
_ac = X_tr_full[:, _idx['a_cent_last']]
_aN = X_tr_full[:, _idx['a_N']]
_aB = X_tr_full[:, _idx['a_B']]

err1 = np.abs(_at - _va * _an).max()
err2 = np.abs(_an**2 - (_at**2 + _ac**2)).max()
err3 = np.abs(_ac**2 - (_aN**2 + _aB**2)).max()
print(f'(1) a_tang_last = va_dot × acc_norm_last  max|err| = {err1:.3e}')
print(f'(2) acc_norm² = a_tang² + a_cent²          max|err| = {err2:.3e}')
print(f'(3) a_cent² = a_N² + a_B²                  max|err| = {err3:.3e}')
# threshold 1e-2: float32 precision (|a|~5 m/s², |a|²~25, accumulation noise ~1e-3 normal)
assert err1 < 1e-3, f'(1) err {err1:.3e} > 1e-3 — fundamental dependency broken'
assert err2 < 1e-2 and err3 < 1e-2, f'(2)/(3) err > 1e-2 — Pythagorean identity broken beyond float32 noise'
print('algebraic identities verified (float32 precision)')

## 셀 6 — Gain ranking 로드 + 7 config feature subset 정의

5-fold mean gain ranking (results/v3_26_gain_ranking.json, seed=42) 기준 nested top-K + dependency-aware 2 config.

In [ ]:
# Gain ranking 출처: results/v3_26_gain_ranking.json (Drive 또는 로컬 .npy 캐시와 함께 업로드 필요)
# 캐시 없으면 5-fold gain ranking 즉시 계산 (no-weight Phase 1, single seed=42)
GAIN_RANKING_PATH = 'results/v3_26_gain_ranking.json'
if not os.path.exists(GAIN_RANKING_PATH):
    print(f'{GAIN_RANKING_PATH} 없음 — 즉시 5-fold gain ranking 계산 (~2분)')
    _folds_g = make_splits(minority_mask_tr.astype(int), seed=42)
    _gain_axis = np.zeros((3, 26), dtype=np.float64)
    for _tr, _va in _folds_g:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr_full[_tr], residual_fren_train[_tr, ax],
                  eval_set=[(X_tr_full[_va], residual_fren_train[_va, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            _gain_axis[ax] += m.booster_.feature_importance(importance_type='gain')
    _gain_mean = (_gain_axis / N_FOLDS).mean(axis=0)
    _order = np.argsort(-_gain_mean)
    gain_meta = {
        'order_by_mean_gain': [FEAT_NAMES_V3_26[int(i)] for i in _order],
        'mean_gain': {FEAT_NAMES_V3_26[int(i)]: float(_gain_mean[int(i)]) for i in _order},
        'note': '5-fold mean gain, no-weight Phase 1, seed=42 (in-notebook recomputed)',
    }
    with open(GAIN_RANKING_PATH, 'w') as f:
        json.dump(gain_meta, f, indent=2)
    print(f'  recomputed and saved: {GAIN_RANKING_PATH}')
else:
    with open(GAIN_RANKING_PATH) as f:
        gain_meta = json.load(f)
    print(f'gain ranking loaded: {GAIN_RANKING_PATH}')

RANK_ORDER = gain_meta['order_by_mean_gain']  # length 26, descending gain
assert len(RANK_ORDER) == 26
assert set(RANK_ORDER) == set(FEAT_NAMES_V3_26)
print(f'Top-5: {RANK_ORDER[:5]}')
print(f'Bottom-6: {RANK_ORDER[-6:]}')

# 7 config feature subset 정의
K5_FEATS    = RANK_ORDER[:5]
K10_FEATS   = RANK_ORDER[:10]
K15_FEATS   = RANK_ORDER[:15]
K20_FEATS   = RANK_ORDER[:20]
K26_FEATS   = list(FEAT_NAMES_V3_26)
K5_INDEP_FEATS   = [f for f in K5_FEATS if f != 'va_dot']
K_TIED_MIN_FEATS = [f for f in FEAT_NAMES_V3_26 if f not in {'va_dot', 'acc_norm_last'}]

CONFIGS = {
    'K5_indep':   K5_INDEP_FEATS,
    'K5':         K5_FEATS,
    'K10':        K10_FEATS,
    'K15':        K15_FEATS,
    'K20':        K20_FEATS,
    'K_tied_min': K_TIED_MIN_FEATS,
    'K26':        K26_FEATS,
}

# Sanity
assert len(K5_INDEP_FEATS) == 4, f'K5_indep should be 4 feat, got {len(K5_INDEP_FEATS)}'
assert len(K_TIED_MIN_FEATS) == 24, f'K_tied_min should be 24 feat, got {len(K_TIED_MIN_FEATS)}'
assert 'va_dot' not in K5_INDEP_FEATS and 'va_dot' not in K_TIED_MIN_FEATS
assert 'acc_norm_last' not in K_TIED_MIN_FEATS
assert set(K5_FEATS).issubset(K10_FEATS) and set(K10_FEATS).issubset(K15_FEATS)
assert set(K15_FEATS).issubset(K20_FEATS) and set(K20_FEATS).issubset(K26_FEATS)

print(f'\n{"Config":<12} {"n_feat":>6}  features')
for cn, fs in CONFIGS.items():
    print(f'{cn:<12} {len(fs):>6}  {fs}')

## 셀 7 — Per-config X matrix (column 선택)

In [ ]:
def build_X_subset(feat_list):
    cols = [FEAT_NAMES_V3_26.index(n) for n in feat_list]
    return X_tr_full[:, cols], X_ts_full[:, cols]

X_per_config = {}
for cn, fs in CONFIGS.items():
    Xtr, Xts = build_X_subset(fs)
    X_per_config[cn] = (Xtr, Xts)
    print(f'  {cn}: Xtr {Xtr.shape}, Xts {Xts.shape}')

## 셀 8 — run_one_config_seed (T-AX C1 protocol bit-exact)

Phase 1 no-weight Frenet target → OOF e_cart → Phase 2 axis-wise W1 (σ_C1, max_w=2.5).

In [ ]:
def inverse_fren_to_cart(resid_fren, eL, eN, eB):
    return (resid_fren[:, 0:1] * eL + resid_fren[:, 1:2] * eN + resid_fren[:, 2:3] * eB)


def run_one_config_seed(config_name, X_tr, X_ts, seed, sigmas=SIGMA_C1, max_w=BEST_MAX_W):
    folds = make_splits(minority_mask_tr.astype(int), seed=seed)

    # ── Phase 1: no-weight (Frenet target) ──
    oof_resid_p1 = np.full((10000, 3), np.nan, dtype=np.float32)
    for tr_idx, val_idx in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr[tr_idx], residual_fren_train[tr_idx, ax],
                  eval_set=[(X_tr[val_idx], residual_fren_train[val_idx, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof_resid_p1[val_idx, ax] = m.predict(X_tr[val_idx]).astype(np.float32)

    oof_cart_p1 = inverse_fren_to_cart(oof_resid_p1, eL_tr, eN_tr, eB_tr)
    pred_p1 = base_train + oof_cart_p1
    oof_e = np.linalg.norm(y_train - pred_p1, axis=1).astype(np.float32)
    HR_p1 = hit_rate(pred_p1, y_train)

    # ── Phase 2: axis-wise W1 ──
    w_axis_arr = w_axis_3(oof_e, sigmas, max_w)
    oof_resid_p2 = np.full((10000, 3), np.nan, dtype=np.float32)
    test_resid_p2 = np.zeros((10000, 3), dtype=np.float32)
    axis_mae_p2 = np.zeros(3, dtype=np.float64)
    for tr_idx, val_idx in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr[tr_idx], residual_fren_train[tr_idx, ax],
                  sample_weight=w_axis_arr[tr_idx, ax],
                  eval_set=[(X_tr[val_idx], residual_fren_train[val_idx, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            pred_val = m.predict(X_tr[val_idx]).astype(np.float32)
            oof_resid_p2[val_idx, ax] = pred_val
            test_resid_p2[:, ax] += m.predict(X_ts).astype(np.float32) / N_FOLDS

    # Per-axis MAE (Frenet residual space)
    for ax in range(3):
        axis_mae_p2[ax] = np.abs(oof_resid_p2[:, ax] - residual_fren_train[:, ax]).mean()

    oof_cart_p2  = inverse_fren_to_cart(oof_resid_p2, eL_tr, eN_tr, eB_tr)
    test_cart_p2 = inverse_fren_to_cart(test_resid_p2, eL_ts, eN_ts, eB_ts)
    pred_p2 = base_train + oof_cart_p2
    HR_p2 = hit_rate(pred_p2, y_train)
    HR_p2_major = hit_rate(pred_p2[~minority_mask_tr], y_train[~minority_mask_tr])
    HR_p2_minor = hit_rate(pred_p2[ minority_mask_tr], y_train[ minority_mask_tr])

    return dict(
        config=config_name, seed=seed,
        HR_p1=HR_p1, HR_p2_overall=HR_p2, HR_p2_major=HR_p2_major, HR_p2_minor=HR_p2_minor,
        axis_mae_p2=axis_mae_p2.tolist(),
        oof_resid_cart=oof_cart_p2, test_resid_cart=test_cart_p2,
    )

print('helper ready')

## 셀 9 — Multi-seed 학습 (7 config × 3 seed = 21 runs)

540 model (K=26은 매번 다시 학습 — 캐시 재사용 옵션은 셀 끝에 메모). ~33~38분 Colab T4.

In [ ]:
all_results = {}
test_preds = {c: [] for c in CONFIGS}
oof_preds  = {c: [] for c in CONFIGS}

_t_all = time.time()
for cname, fs in CONFIGS.items():
    Xtr, Xts = X_per_config[cname]
    for seed in SEEDS:
        t0 = time.time()
        r = run_one_config_seed(cname, Xtr, Xts, seed)
        all_results[(cname, seed)] = r
        test_preds[cname].append(r['test_resid_cart'])
        oof_preds[cname].append(r['oof_resid_cart'])
        np.save(f'results/topk_e29_oof_{cname}_seed{seed}.npy', r['oof_resid_cart'])
        np.save(f'results/topk_e29_test_{cname}_seed{seed}.npy', r['test_resid_cart'])
        print(f'  {cname:<12} (n_feat={len(fs):>2}) seed={seed:3d}: '
              f'P1={r["HR_p1"]:.4f}  P2={r["HR_p2_overall"]:.4f}  '
              f'maj={r["HR_p2_major"]:.4f}  min={r["HR_p2_minor"]:.4f}  '
              f'({time.time()-t0:.0f}s)')

print(f'\nMulti-seed 학습 완료 (총 {time.time()-_t_all:.0f}s)')

## 셀 10 — Aggregate + Sanity + 3-gate + 4th guard

Sanity: K26 OOF ≈ T-AX C1 reference (0.6622). 3-gate: 6 신규 config vs K26 (this run's control).

In [ ]:
def aggregate(cfg):
    rs = [all_results[(cfg, s)] for s in SEEDS]
    o = np.array([r['HR_p2_overall'] for r in rs])
    M = np.array([r['HR_p2_major'] for r in rs])
    m = np.array([r['HR_p2_minor'] for r in rs])
    p1 = np.array([r['HR_p1'] for r in rs])
    mae = np.array([r['axis_mae_p2'] for r in rs])  # (3 seed, 3 axis)
    return dict(
        n_feat=len(CONFIGS[cfg]),
        overall_mean=float(o.mean()), overall_std=float(o.std(ddof=1)),
        major_mean=float(M.mean()),   major_std=float(M.std(ddof=1)),
        minor_mean=float(m.mean()),   minor_std=float(m.std(ddof=1)),
        p1_mean=float(p1.mean()),     p1_std=float(p1.std(ddof=1)),
        axis_mae_mean=mae.mean(axis=0).tolist(),
        per_seed_overall=o.tolist(),
    )

agg = {c: aggregate(c) for c in CONFIGS}

print(f'{"Config":<12} {"n":>3}  {"P1 mean±std":<18} {"P2 overall±std":<20} {"P2 major±std":<20} {"P2 minor±std":<20}  rL_mae  rN_mae  rB_mae')
print('-'*150)
for c in CONFIGS:
    a = agg[c]
    mae = a['axis_mae_mean']
    print(f'{c:<12} {a["n_feat"]:>3}  '
          f'{a["p1_mean"]:.4f} ± {a["p1_std"]:.4f}  '
          f'{a["overall_mean"]:.4f} ± {a["overall_std"]:.4f}  '
          f'{a["major_mean"]:.4f} ± {a["major_std"]:.4f}  '
          f'{a["minor_mean"]:.4f} ± {a["minor_std"]:.4f}  '
          f'{mae[0]*1000:.3f}  {mae[1]*1000:.3f}  {mae[2]*1000:.3f}')

print(f'\nPer-seed P2 overall:')
for c in CONFIGS:
    vals = [all_results[(c, s)]['HR_p2_overall'] for s in SEEDS]
    print(f'  {c:<12}: ' + '  '.join(f'seed={s} {h:.4f}' for s, h in zip(SEEDS, vals)))

# Sanity: K26 P2 overall should match T-AX C1 reference (0.6622)
print(f'\n=== Sanity (K26 ≈ T-AX C1 OOF {C1_REF_OVERALL:.4f}) ===')
k26 = agg['K26']
sanity_p1 = abs(k26['p1_mean'] - C1_REF_P1) <= 0.003
sanity_overall = abs(k26['overall_mean'] - C1_REF_OVERALL) <= 0.003
print(f'  K26 P1   = {k26["p1_mean"]:.4f}, ref {C1_REF_P1:.4f}, diff {k26["p1_mean"]-C1_REF_P1:+.4f}  '
      f'{"O" if sanity_p1 else "X (protocol drift)"}')
print(f'  K26 OOF  = {k26["overall_mean"]:.4f}, ref {C1_REF_OVERALL:.4f}, diff {k26["overall_mean"]-C1_REF_OVERALL:+.4f}  '
      f'{"O" if sanity_overall else "X (protocol drift)"}')

# 3-gate: 6 신규 config vs K26 (this run's control)
TEST_CONFIGS = [c for c in CONFIGS if c != 'K26']
n_test = len(TEST_CONFIGS)  # 6
bonferroni = math.sqrt(n_test)
FOURTH_GUARD_FACTOR = math.sqrt(2)  # EXP #28

print(f'\n=== 3-gate strict + 4th guard (vs K26, Bonferroni √{n_test} = {bonferroni:.3f}, 4th guard √2 = {FOURTH_GUARD_FACTOR:.3f}) ===')
gate_results = {}
for cn in TEST_CONFIGS:
    a = agg[cn]
    d_o = a['overall_mean'] - k26['overall_mean']
    d_M = a['major_mean']   - k26['major_mean']
    d_m = a['minor_mean']   - k26['minor_mean']
    combined_std = (a['overall_std']**2 + k26['overall_std']**2) ** 0.5
    g1_thr = combined_std * bonferroni
    g4_thr = combined_std * FOURTH_GUARD_FACTOR
    g1 = d_o > g1_thr
    g2 = d_M > -0.002
    g3 = d_m > -0.005
    g4 = d_o > g4_thr  # 4th guard (LB submission gate)
    pass3 = bool(g1 and g2 and g3 and sanity_p1 and sanity_overall)
    lb_ok = bool(g4 and g2 and g3 and sanity_p1 and sanity_overall)
    gate_results[cn] = dict(
        n_feat=a['n_feat'],
        d_overall=float(d_o), d_major=float(d_M), d_minor=float(d_m),
        g1_thr=float(g1_thr), g4_thr=float(g4_thr),
        g1=bool(g1), g2=bool(g2), g3=bool(g3), g4=bool(g4),
        pass3=pass3, lb_eligible=lb_ok,
    )
    print(f'  {cn:<12} (n={a["n_feat"]:>2}): '
          f'Δo={d_o:+.4f} (g1_thr {g1_thr:+.4f}, g4_thr {g4_thr:+.4f})  '
          f'ΔM={d_M:+.4f}  Δm={d_m:+.4f}  '
          f'G1={"O" if g1 else "X"} G2={"O" if g2 else "X"} G3={"O" if g3 else "X"} G4={"O" if g4 else "X"}  '
          f'pass3={"O" if pass3 else "X"} LB={"O" if lb_ok else "X"}')

# 채택 결정
passed_strict = [cn for cn in TEST_CONFIGS if gate_results[cn]['pass3']]
lb_eligible   = [cn for cn in TEST_CONFIGS if gate_results[cn]['lb_eligible']]
if not lb_eligible:
    chosen = None
    print(f'\n채택 없음 — 4th guard (Δo > {k26["overall_std"]*2.0*FOURTH_GUARD_FACTOR:+.4f} 근처) 통과 config 없음. lever 폐기 후보.')
else:
    # LB eligible 중 overall best
    chosen = max(lb_eligible, key=lambda c: agg[c]['overall_mean'])
    pass_label = 'strict (G1)' if chosen in passed_strict else '4th guard only (LB)'
    print(f'\n채택: {chosen} ({pass_label}, overall {agg[chosen]["overall_mean"]:.4f})')

## 셀 11 — Algebraic ablation 분석 (K5_indep vs K5, K_tied_min vs K26)

Plan §4.14 핵심 finding: va_dot이 a_tang_last에 흡수되는지 + algebraic 잉여 제거 효과.

In [ ]:
def delta_pair(name_a, name_b):
    """name_a vs name_b (Δ = a - b)."""
    a = agg[name_a]; b = agg[name_b]
    cstd = (a['overall_std']**2 + b['overall_std']**2) ** 0.5
    return dict(
        d_overall=a['overall_mean'] - b['overall_mean'],
        d_major=a['major_mean'] - b['major_mean'],
        d_minor=a['minor_mean'] - b['minor_mean'],
        combined_std=cstd,
        signal_zone=(abs(a['overall_mean'] - b['overall_mean']) > cstd),
    )

alg = {
    'K5_indep_vs_K5':       delta_pair('K5_indep', 'K5'),
    'K_tied_min_vs_K26':    delta_pair('K_tied_min', 'K26'),
}

print('=== Algebraic ablation ===')
print(f'\n1) K5_indep (4 feat) vs K5 (5 feat) — va_dot drop 효과')
d = alg['K5_indep_vs_K5']
print(f'   Δoverall = {d["d_overall"]:+.4f} (combined_std {d["combined_std"]:.4f}, signal_zone={d["signal_zone"]})')
print(f'   Δmajor   = {d["d_major"]:+.4f}')
print(f'   Δminor   = {d["d_minor"]:+.4f}')
if abs(d['d_overall']) < d['combined_std']:
    print('   → va_dot은 a_tang_last에 흡수 (redundant) 정량 확인')
elif d['d_overall'] > 0:
    print('   → K5_indep > K5 (va_dot drop이 OOF 향상) — va_dot이 noise였음')
else:
    print('   → K5_indep < K5 (va_dot은 algebraic dependency에도 LGBM에 추가 split utility)')

print(f'\n2) K_tied_min (24 feat) vs K26 (26 feat) — algebraic 잉여 {{va_dot, acc_norm_last}} 제거 효과')
d = alg['K_tied_min_vs_K26']
print(f'   Δoverall = {d["d_overall"]:+.4f} (combined_std {d["combined_std"]:.4f}, signal_zone={d["signal_zone"]})')
print(f'   Δmajor   = {d["d_major"]:+.4f}')
print(f'   Δminor   = {d["d_minor"]:+.4f}')
if d['d_overall'] > d['combined_std']:
    print('   → algebraic redundancy가 V3 26 성능 깎고 있음 정량 확인 (가설 H_K 강한 지지)')
elif abs(d['d_overall']) < d['combined_std']:
    print('   → algebraic 잉여도 net 0 (LGBM이 redundant view에서 split convenience 추출)')
else:
    print('   → K_tied_min < K26 (잉여 feature가 LGBM에 net positive utility)')

# Save algebraic ablation
with open('results/topk_e29_algebraic.json', 'w', encoding='utf-8') as f:
    json.dump({k: {kk: float(vv) if isinstance(vv, (int, float, np.floating)) else bool(vv) for kk, vv in v.items()}
                for k, v in alg.items()}, f, indent=2)
print('\nresults/topk_e29_algebraic.json 저장')

## 셀 12 — Top-K curve 분석 (nested 5 점)

K=5/10/15/20/26 overall/major/minor 3 곡선 + axis별 MAE 분해.

In [ ]:
nested = ['K5', 'K10', 'K15', 'K20', 'K26']
Ks = [len(CONFIGS[c]) for c in nested]

print(f'{"K":>3}  {"overall":>10}  {"major":>10}  {"minor":>10}  {"rL_mae":>8}  {"rN_mae":>8}  {"rB_mae":>8}')
for c, K in zip(nested, Ks):
    a = agg[c]
    mae = a['axis_mae_mean']
    print(f'{K:>3}  {a["overall_mean"]:>10.4f}  {a["major_mean"]:>10.4f}  {a["minor_mean"]:>10.4f}  '
          f'{mae[0]*1000:>6.3f}mm  {mae[1]*1000:>6.3f}mm  {mae[2]*1000:>6.3f}mm')

# Peak 위치 식별
ovs = [agg[c]['overall_mean'] for c in nested]
majs = [agg[c]['major_mean'] for c in nested]
mins = [agg[c]['minor_mean'] for c in nested]
peak_overall = nested[int(np.argmax(ovs))]
peak_major   = nested[int(np.argmax(majs))]
peak_minor   = nested[int(np.argmax(mins))]
print(f'\nPeak by metric:')
print(f'  overall peak = {peak_overall} ({max(ovs):.4f})')
print(f'  major   peak = {peak_major} ({max(majs):.4f})')
print(f'  minor   peak = {peak_minor} ({max(mins):.4f})')

# 단조 vs peak 패턴
monotone_up = all(ovs[i] <= ovs[i+1] for i in range(len(ovs)-1))
monotone_down = all(ovs[i] >= ovs[i+1] for i in range(len(ovs)-1))
if monotone_up:
    pattern = 'K↑ ⇒ OOF↑ 단조 (V3 26 full이 best — 가설 H_K 기각 방향)'
elif monotone_down:
    pattern = 'K↑ ⇒ OOF↓ 단조 (작은 subset이 best — 가설 H_K 강한 지지)'
elif peak_overall in ('K10', 'K15'):
    pattern = f'중간 peak ({peak_overall}) — 가설 H_K 부분 지지, plateau noise 확인'
else:
    pattern = '비단조 — ranking instability 또는 carrier 비대칭 의심'
print(f'\n패턴: {pattern}')

## 셀 13 — Submission 생성 (lb_eligible config 모두 + 채택 우선)

In [ ]:
def make_submission(test_resid_avg, name):
    pred_test_abs = base_test + test_resid_avg
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()
    df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    path = f'submission_topk_{name}.csv'
    df.to_csv(path, index=False)
    print(f'  {path}: x[{pred_test_abs[:,0].min():.2f},{pred_test_abs[:,0].max():.2f}] '
          f'y[{pred_test_abs[:,1].min():.2f},{pred_test_abs[:,1].max():.2f}] '
          f'z[{pred_test_abs[:,2].min():.2f},{pred_test_abs[:,2].max():.2f}]')
    return path

sub_paths = {}
if chosen is not None:
    # 채택 config + 다른 lb_eligible 모두 생성 (cherry-pick 위해)
    for c in lb_eligible:
        avg = np.mean(test_preds[c], axis=0).astype(np.float32)
        p = make_submission(avg, f'{c}_ms3')
        sub_paths[c] = p
    print(f'\n채택 {chosen} → 우선 제출: {sub_paths[chosen]}')
    if len(lb_eligible) > 1:
        print(f'  기타 lb_eligible: {[c for c in lb_eligible if c != chosen]} (보조 자료)')
else:
    print('채택 없음 — submission 생성 skip (4th guard 미충족)')

## 셀 14 — Meta 저장 + Drive 복사 + 로컬 다운로드

In [ ]:
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, np.ndarray): return x.tolist()
    return x

agg_safe = {k: {kk: safe(vv) for kk, vv in v.items()} for k, v in agg.items()}
gate_safe = {k: {kk: safe(vv) for kk, vv in v.items()} for k, v in gate_results.items()}

meta = {
    'protocol': 'EXP #29 Top-K Feature Subset Ablation (T-AX C1 protocol bit-exact)',
    'seeds': SEEDS, 'n_folds': N_FOLDS,
    'configs': {k: list(v) for k, v in CONFIGS.items()},
    'configs_n_feat': {k: len(v) for k, v in CONFIGS.items()},
    'sigma_C1_axis': list(SIGMA_C1),
    'max_w': BEST_MAX_W,
    'minority_threshold': MINORITY_THRESHOLD,
    't_ax_c1_reference_exp17': {'overall': C1_REF_OVERALL, 'overall_std': C1_REF_OVERALL_STD,
                                  'major': C1_REF_MAJOR, 'minor': C1_REF_MINOR, 'p1': C1_REF_P1},
    'gain_ranking_source': GAIN_RANKING_PATH,
    'rank_order': RANK_ORDER,
    'aggregate': agg_safe,
    'sanity_p1_pass': bool(sanity_p1),
    'sanity_overall_pass': bool(sanity_overall),
    'gate_results_vs_K26': gate_safe,
    'bonferroni_factor': float(bonferroni),
    'fourth_guard_factor': float(FOURTH_GUARD_FACTOR),
    'chosen': chosen,
    'lb_eligible': lb_eligible,
    'passed_strict_g1': passed_strict,
    'submissions': sub_paths,
    'topk_curve': {
        'K': Ks,
        'overall': ovs,
        'major': majs,
        'minor': mins,
        'peak_overall': peak_overall,
        'peak_major': peak_major,
        'peak_minor': peak_minor,
        'pattern': pattern,
    },
    'algebraic_ablation': {k: {kk: safe(vv) for kk, vv in v.items()} for k, v in alg.items()},
}
with open('results/topk_e29_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('results/topk_e29_meta.json 저장')

try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    # 노트북 자체 백업
    for nb_name in ['hr_aware_topk_e29.ipynb']:
        if os.path.exists(nb_name):
            !cp {nb_name} {DRIVE_BASE}/
    for p in sub_paths.values():
        !cp {p} {DRIVE_BASE}/
        files.download(p)
    print('Drive 복사 + 다운로드 완료')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')